# Café Analytics - Descriptive Analysis

In [ ]:
import pandas as pd
import warnings
import numpy as np
from datetime import datetime, date, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.stats.proportion import proportion_confint

In [ ]:
# Ignore all warnings in output
warnings.filterwarnings('ignore')

In [ ]:
# Set the maximum column width for pandas DataFrame display
pd.set_option('max_colwidth', 2000)

## Revenue Trend Analysis

In [ ]:
# Load the cleaned full data parquet file into a pandas DataFrame
df = pd.read_parquet(
    './cafe_processed_files/cafe_full_mx.parquet',
)

In [ ]:
df_revenue = df[
    ['order_id', 'customer_id', 'order_price', 'order_time']
].drop_duplicates().reset_index(drop=True)

In [ ]:
df_revenue.head()

In [ ]:
df_revenue['order_year'] = (
    df_revenue['order_time'].dt.year
)
df_revenue['order_month'] = (
    df_revenue['order_time'].dt.month
)
df_revenue['order_day'] = (
    df_revenue['order_time'].dt.day
)
df_revenue['order_dow'] = (
    df_revenue['order_time'].dt.dayofweek
)
df_revenue['order_hour'] = (
    df_revenue['order_time'].dt.hour
)

In [ ]:
df_revenue['FY'] = (
    df_revenue.apply(
        lambda x: 'FY' + str(x['order_year'])[-2:] if x['order_month'] < 7
        else 'FY' + str(x['order_year'] + 1)[-2:],
        axis=1,
    )
)

In [ ]:
df_revenue.head()

### Monthly Revenue Trend

In [ ]:
monthly = (
    df_revenue[
        ['order_year', 'order_month', 'order_price']
    ].groupby(['order_year', 'order_month']).sum()
    .reset_index(drop=False)
)

In [ ]:
monthly.head()

In [ ]:
# Plot monthly revenue trend
fig, axes = plt.subplots(nrows=2, figsize=(10, 12))

for year in sorted(monthly['order_year'].unique()):
    month_data = monthly[
        monthly['order_year'] == year
    ]
    Y = month_data['order_price']
    X = month_data['order_month']
    
    axes[1].plot(X, Y, label=year, marker='.')
    axes[0].bar(
        x=year,
        height=month_data['order_price'].sum(), 
        label=year,
    )

axes[0].set_title('Online Sales Trend Over Years')
axes[0].legend(loc='upper left', bbox_to_anchor=(1.05, 1))
axes[0].set_xlabel('')
axes[0].set_ylabel('Online Sales (AUD)')
axes[1].set_title('Online Sales Trends by Years and Months')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Online Sales (AUD)')
axes[1].set_xticks(range(1, 13, 1))
plt.show()

* The bar chart above illustrates a **consistent rise** in business online sales from 2020 to 2023.
* Online sales in 2020 and 2021 consistently outperformed their corresponding previous year across all months, showing a notable increase during the winter season (June to August) and peaking in September before declining.
* In 2022, online sales generally stabilized over the course of the year but stayed higher than in 2021, except during the period from August to October. 
* Similary, online sales in 2023 remained stable overall but generally exceeded those of 2022 throughout the year.
* However, neither 2022 nor 2023 reached the September peak achieved in 2020 and 2021.

Question to investigate:<br><br>
**1) What factors contributed to the surge in sales from June to September in 2020 and 2021?**<br>
**2) Why did sales in 2022 and 2023 fail to reach the same peak in September?**<br>

Possible reasons: 

The **COVID pandemic lockdowns** likely boosted online sales, making the 2020 and 2021 figures more reflective of total sales (including both online and in-store). In contrast, the stablized sales in 2022 and 2023 may represent online sales only, as in-store shopping resumed.

As seen in the 2020 and 2021 figures, the café's online sales typically rise during the colder months from June to September in Australia, followed by a decline as the weather warms.

**TBC...**

### Most Profitable Day of the Week

In [ ]:
total_revenue_per_day = (
    df_revenue[
        ['order_year', 'order_month', 'order_day', 
         'order_dow', 'order_price']
    ].groupby([
        'order_year', 'order_month', 
        'order_day', 'order_dow'
    ]).sum().reset_index(drop=False)
)

total_revenue_per_day.head()

In [ ]:
dow_revenue = (
    total_revenue_per_day[
        ['order_dow', 'order_price']
    ].groupby('order_dow').mean()
    .reset_index(drop=False)
)

dow_revenue

In [ ]:
day_of_week = {
    0: 'Mon',
    1: 'Tue',
    2: 'Wed',
    3: 'Thu',
    4: 'Fri',
    5: 'Sat',
    6: 'Sun',
}

In [ ]:
# Plot average daily revenue over a week
fig, ax = plt.subplots(figsize=(6, 4))

dow_revenue.plot.bar(
    x='order_dow', y='order_price', 
    color='#D3D0C9', ax=ax
)

# Highlight the most profitable day of the week
top_dow = dow_revenue[
    (
        dow_revenue['order_price'] 
        == dow_revenue['order_price'].max() 
    )
]
ax.bar(
    x=top_dow['order_dow'],
    height=top_dow['order_price'],
    color='orange',
    width=0.5,
)

plt.xticks(
    ticks=list(day_of_week.keys()),
    labels=list(day_of_week.values()), 
    rotation=0,
)
plt.xlabel('')
ax.get_legend().remove()
ax.set_ylabel('Online Sales (AUD)')
plt.show()

As shown in the bar charts above, the average daily online sales generally remained steady from Mondays to Thursdays, but rose higher during Fridays and weekends, with a peak on **Saturdays** and **Sundays**.

### Peak Revenue Hours

In [ ]:
df_revenue.head()

In [ ]:
# Group sales revenue by year, month, day and hour
total_revenue_per_hour = (
    df_revenue[
        ['order_year', 'order_month', 'order_day', 
         'order_hour', 'order_price']
    ].groupby([
        'order_year', 'order_month', 
        'order_day', 'order_hour'
    ]).sum().reset_index(drop=False)
)

In [ ]:
print(total_revenue_per_hour.shape)
total_revenue_per_hour.head()

In [ ]:
# Display operating hours
sorted(total_revenue_per_hour['order_hour'].unique())

Some of order-taking hours seem unusual for a café. Let's check whether there are any outliers (irregular ordering hours) among them.

#### Identify and Remove Outliers

In [ ]:
# Plot the KDE distribution of records across operating hours
fig, ax = plt.subplots()
sns.kdeplot(
    x=total_revenue_per_hour['order_hour'],
    bw_adjust=1.6,  
    color='gray',
    ax=ax,
)
ax.set_xlabel('Order Hour')
ax.set_xticks(
    range(
        0,
        total_revenue_per_hour['order_hour'].max() + 1,
        1,
    )
)
plt.show()

According to the Kernel Density Estimation (KDE) plot above, the ordering hours in most days are between 5AM to 3PM. Let's confirm this using a boxplot and identify any potential outliers.

In [ ]:
# Draw a boxplot to identify outliers of order hours
fig, ax = plt.subplots()
sns.boxplot(
    y=total_revenue_per_hour['order_hour'],
    whis=1.5,    # Set the whiskers to extend up to 1.5 times the interquartile range (IQR)
    ax=ax,
)
plt.title("Data Point Distribution of Order Hours")
plt.ylabel('Operating Hour')
plt.yticks(
    range(
        total_revenue_per_hour['order_hour'].min(),
        total_revenue_per_hour['order_hour'].max() + 1,
        1,
    )
)
plt.show()

As shown in the boxplot above, it appears that the ordering hours extending past 5 PM have been identified as outliers.

In [ ]:
# Calculate the 1st quartile
Q1 = total_revenue_per_hour['order_hour'].quantile(0.25)
# Calculate the 3rd quartile
Q3 = total_revenue_per_hour['order_hour'].quantile(0.75)
# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate the lower bound where the lower whisker ends
lower_bound = Q1 - 1.5 * IQR
# Calculate the upper bound where the upper whisker ends
upper_bound = Q3 + 1.5 * IQR

print(lower_bound)
print(upper_bound)

Data points falling below the lower bound or above the upper bound are considered **outliers**. Let's apply the lower and upper bounds to filter out irregular ordering hours, which only occurred on a few days in the business history, and determine the actual online ordering hours.

In [ ]:
# Filter out irregular ordering hours
total_revenue_per_hour = total_revenue_per_hour[
    (total_revenue_per_hour['order_hour'] >= lower_bound)
    & (total_revenue_per_hour['order_hour'] <= upper_bound)
]

In [ ]:
# Print the actual online ordering hours
sorted(total_revenue_per_hour['order_hour'].unique())

After filtering out the irregular ordering hours, we can see that the café typically accepts orders from **5 AM to 5 PM** on regular business days.

In [ ]:
# Calculate the average online sales revenue per hour
hourly_revenue = (
    total_revenue_per_hour[
        ['order_hour', 'order_price']
    ].groupby('order_hour').mean()
    .reset_index(drop=False)
)

hourly_revenue

In [ ]:
# Calculate the average revenue per hour across all days
avg_revenue_per_hour = total_revenue_per_hour['order_price'].mean()
print(avg_revenue_per_hour)

In [ ]:
# Plot average hourly revenue across hours
plt.bar(
    x=hourly_revenue['order_hour'],
    height=hourly_revenue['order_price'],
    color='#D3D0C9',
)
# Plot a constant line representing average revenue per 
# hour across all days
plt.plot(
    range(0, 24, 1),
    [avg_revenue_per_hour] * 24,
    linestyle='--',
    linewidth=0.5,
    c='black',
)

# Highlight the peak revenue hours
top_revenue = hourly_revenue[
    hourly_revenue['order_price'] > avg_revenue_per_hour
]

plt.bar(
    x=top_revenue['order_hour'],
    height=top_revenue['order_price'],
    color='orange',
)
plt.title('Online Sales by Hours')
plt.xlabel('Hour')
plt.ylabel('Online Sales (AUD)')
plt.xticks(range(0, 24, 1))
plt.show()

The peak revenue hours that drive online sales above average are typically between **7 AM and 10 AM**, with the highest sales occurring at **8AM**.

### Revenue and Order Value per Customer

In [ ]:
df_revenue.head()

In [ ]:
cust_revenue = df_revenue[
    ['customer_id', 'order_year', 'order_month', 'order_id', 'order_price']
].sort_values(
    ['customer_id', 'order_year', 'order_month', 'order_id', 'order_price'],
    ascending=[True, True, True, True, True],
).reset_index(drop=True)

cust_revenue.head(10)

In [ ]:
# Create a DataFrame showing each customer's total spending
# over the entire business history in Descending order
total_spending_per_cust = (
    cust_revenue[
        ['customer_id', 'order_price']
    ].groupby('customer_id')
    .sum().sort_values('order_price', ascending=False)
    .rename(columns={'order_price': 'total_spending'})
    .reset_index(drop=False)
)
# Calculate the average total spending
avg_total_spending = total_spending_per_cust['total_spending'].mean()

# Create a DataFrame showing the average spending per order by each
# customer in Descending order
spending_per_order_per_cust = (
    cust_revenue[
        ['customer_id', 'order_price']
    ].groupby('customer_id')
    .mean().sort_values('order_price', ascending=False)
    .rename(columns={'order_price': 'spending_per_order'})
    .reset_index(drop=False)
)
# Calculate average customer spending per order
avg_spending_per_order = (
    spending_per_order_per_cust['spending_per_order'].mean()
)

# Create a DataFrame showing each customer's order frequency over 
# the entire business history in Descending order
order_frequency_per_cust = (
    cust_revenue[
        ['customer_id', 'order_price']
    ].groupby('customer_id')
    .count().sort_values('order_price', ascending=False)
    .rename(columns={'order_price': 'order_freq'})
    .reset_index(drop=False)
)
# Calculate average order frequency
avg_order_frequency = (
    order_frequency_per_cust['order_freq'].mean()
)

In [ ]:
print(
    "The average customer total spending over their "
    "customer lifespan: ${}"
    .format(
        round(avg_total_spending, 2),
    )
)
total_spending_per_cust.head()

In [ ]:
print(
    "The average customer spending per order: ${}".format(
        round(avg_spending_per_order, 2)
    )
)
spending_per_order_per_cust.head()

In [ ]:
print(
    "The average order frequency: {}".format(
        round(avg_order_frequency)
    )
)
order_frequency_per_cust.head()

In [ ]:
top10_total_spending = total_spending_per_cust.head(10)

fig, ax = plt.subplots()

sns.barplot(
    data=top10_total_spending,
    x='total_spending',
    y='customer_id',
    orient='h',
    order=top10_total_spending['customer_id'],
    ax=ax,
)

ax.set_title('Top 10 Revenue-driving Customers')
ax.set_ylabel('Customer ID')
ax.set_xlabel('Online Sales Revenue (AUD)')

plt.show()

In [ ]:
top10_order_freq = order_frequency_per_cust.head(10)

fig, ax = plt.subplots()

sns.barplot(
    data=top10_order_freq,
    x='order_freq',
    y='customer_id',
    orient='h',
    order=top10_order_freq['customer_id'],
    ax=ax,
)

ax.set_title('Top 10 Frequent Buyers')
ax.set_ylabel('Customer ID')
ax.set_xlabel('Number of Orders')

plt.show()

In [ ]:
top10_spenders = top10_total_spending['customer_id']

top10_frequent_buyers = top10_order_freq['customer_id']

print(
    '{} out of the top 10 frequent buyers are also '
    'among the top 10 big spenders.'.format(
        len(
            [
                cust for cust in list(top10_frequent_buyers) 
                if cust in list(top10_spenders)
            ]
        )
    )
)

In [ ]:
top10_spending_per_order = spending_per_order_per_cust.head(10)

fig, ax = plt.subplots()

sns.barplot(
    data=top10_spending_per_order,
    x='spending_per_order',
    y='customer_id',
    orient='h',
    order=top10_spending_per_order['customer_id'],
    ax=ax,
)

ax.set_title('Top 10 Customers with Highest Spending per Order')
ax.set_ylabel('Customer ID')
ax.set_xlabel('Average Spending Per Order (AUD)')

plt.show()

In [ ]:
top10_avg_spenders = top10_spending_per_order['customer_id']

print(
    '{} out of the top 10 customers with highest spending per order are also '
    'among the top 10 big spenders.'.format(
        len(
            [
                cust for cust in list(top10_avg_spenders) 
                if cust in list(top10_spenders)
            ]
        )
    )
)

In [ ]:
order_freq = order_frequency_per_cust[
    order_frequency_per_cust['customer_id'].isin(
        top10_spenders
    )
]
spending_per_order =  spending_per_order_per_cust[
    spending_per_order_per_cust['customer_id'].isin(
        top10_spenders
    )
]
spending_per_order.merge(
    order_freq,
    how='inner',
    on='customer_id',
)

#### Discuss Insights:

7 out of the top 10 highest spending customers are also among the top 10 repeat customers; however, none are in the top 10 for highest average spending per order. All of them have placed more orders than the average of 23, but have on average spent less per order than the mean of $17.32.

These observations suggest that customer spending may be more strongly correlated with the total number of orders they placed (order frequency) than with the average spending per order. To verify this, let's examine the **correlation coefficients** between these variables.

#### Determine Correlation Method

There are two common types of correlation coefficients we can calculate: **Pearson Correlation Coefficients** and **Spearman's Rank Correlation Coefficients**. Each method has specific assumptions:
* **Pearson Correlation Coefficients**:
    - The variables are normally distribtued
    - The relationship between the variables is **linear**
* **Spearman's Rank Correlation Coefficients**:
    - The data is not normally distributed
    - The relationship between variables is **monotonic** (consistently increases or decreases but not necessarily at a constant rate) but not necessarily linear
    - The data may contain outliers that could affect Pearson's correlation
    
Let's investigate the customer revenue data to determine which correlation method to use.

In [ ]:
df_corr = total_spending_per_cust.merge(
    order_frequency_per_cust,
    how='inner',
    on='customer_id',
).merge(
    spending_per_order_per_cust,
    how='inner',
    on='customer_id',
)

In [ ]:
# Plot kernel density distribution of customer spending
fig, ax = plt.subplots()
sns.kdeplot(data=df_corr, x='total_spending', ax=ax)
# Plot the mean of revenue by customers as a constant line
ax.plot(
    [df_corr['total_spending'].mean(), df_corr['total_spending'].mean()], 
    [0, 0.0018],
    linestyle='--',
    linewidth=0.8,
)
plt.xlabel('Total Spending by Customers (AUD)')
plt.show()

According to the Kernal Density Estimation plot above, the `total_spending` variable is NOT normally distributed (it is right-skewed) and contains outliers. Therefore, we should use the **Spearman's Rank Correlation Coefficients** to determine the correlations between `total_spending` and `order_freq`, and between `total_spending` and `spending_per_order`.

#### Calculate Spearman's Rank Correlation Coefficients

In [ ]:
# Compute Spearman's Rank Correlation Coefficients matrix 
# between total spending, order frequency and average spending
# per order
df_corr[
    ['total_spending', 'order_freq', 'spending_per_order']
].corr(method='spearman')

The **relatively high** correlation coefficient of **0.848067** between `total_spending` and `order_freq` indicates a **strong positive monotonic relationship** between these two variables. This suggests that as the order frequency increases, the total spending tend to increase as well. The high coefficient value signifies that customers who place orders more often generally contribute more to the total sales.

In contrast, the **relatively low** correlation coefficient (**0.395176**) between `total_spending` and `spending_per_order` indicates a **moderate positive monotonic relationship**. This means there is a weaker correlation between total spending and the average spending per order. While higher spending per order is somewhat associated with higher total sales, the relationship is not as strong or consistent as with the order frequency.

To determine whether the correlations between total spending and order frequency, and between total spending and average spending per order are statistically significant, we can perform **Hypothesis Test** on the Spearman's Rank Correlation Coefficients. This will help us assess the **level of confidence** we can place in the observed relationships.

#### Hypothesis Testing on Spearman's Rank Correlation Coefficients

**Hypotheses for `total_spending` vs. `order_freq`**

* Null Hypothesis ($H_0$): There is no monotonic relationship between `total_spending` and `order_freq` ($\rho = 0$)
* Alternative Hypothesis ($H_1$): There is a statistically significant monotonic relationship between `total_spending` and `order_freq` ($\rho\neq0$)

**Hypotheses for `total_spending` vs. `spending_per_order`**

* Null Hypothesis ($H_0$): There is no monotonic relationship between `total_spending` and `spending_per_order` ($\rho = 0$)
* Alternative Hypothesis ($H_1$): There is a statistically significant monotonic relationship between `total_spending` and `spending_per_order` ($\rho\neq0$)

**Significance Level ($\alpha$)**: 0.05

We will use the **Spearman's Rank Correlation Coefficients** and calculate the corresponding **p-value** to assess the **statistical significance** of the correlation. The p-value represents the probability of observing the data assuming that the null hypothesis is true. By convention, we set the **significance level** ($\alpha$) at **0.05**.

In [ ]:
total_spending = df_corr['total_spending']
order_freq = df_corr['order_freq']

# Calculate Spearman's Rank Correlation and p-value
# for total_spending vs. order_freq
corr_coefficient_ts_of, p_value_ts_of = spearmanr(
    total_spending, order_freq
)

print("Spearman's Rank correlation coefficient between "
      f'total_spending and order_freq: {corr_coefficient_ts_of}')
print(f'p-value: {p_value_ts_of}')

# Scatter plot for total_sales vs. order_count
plt.figure(figsize=(7, 5))
plt.scatter(
    order_freq, 
    total_spending, 
    edgecolors='#d3d3d3',
    s=60,
    linewidth=0.4,
    alpha=0.7,
)
# Adding a LOWESS line to represent a monotonic trend
sns.regplot(
    x=order_freq, 
    y=total_spending, 
    lowess=True, 
    scatter=False, 
    color='black',
    line_kws={
        'linewidth': 1.5,
        'linestyle': '--',
    },
)
plt.xlabel('Number of Orders')
plt.ylabel('Total Spending (AUD)')
plt.title('Total Spending vs. Order Frequency')
plt.show()

In [ ]:
total_spending = df_corr['total_spending']
spending_per_order = df_corr['spending_per_order']

# Calculate Spearman's Rank Correlation and p-value
# for total_spending vs. spending_per_order
corr_coefficient_ts_sp, p_value_ts_sp = spearmanr(
    total_spending, spending_per_order
)

print('Spearman correlation coefficient between '
      f'total_spending and spending_per_order: {corr_coefficient_ts_sp}')
print(f'p-value: {p_value_ts_sp}')

# Scatter plot for total_spending vs spending_per_order
plt.figure(figsize=(7, 5))
plt.scatter(
    spending_per_order, 
    total_spending, 
    edgecolors='#d3d3d3',
    s=60,
    linewidth=0.4,
    alpha=0.7,
)
# Adding a LOWESS line to represent a monotonic trend
sns.regplot(
    x=spending_per_order, 
    y=total_spending, 
    lowess=True, 
    scatter=False, 
    color='black',
    line_kws={
        'linewidth': 1.5,
        'linestyle': '--',
    },
)
plt.xlabel('Average Spending Per Order (AUD)')
plt.ylabel('Total Spending (AUD)')
plt.title('Total Customer Spending vs. Average Order Value')
plt.show()

**Results and Conclusion**:

* **Total Customer Spending vs. Order Frequency**:
    - Spearman's Rank Correlation Coefficient: **0.848**
        * Indicates a **strong positive monotonic relationship** between total customer spending and order frequency.
    - P-value: **0.0**
        * Since the p-value < 0.05, we **reject the null hypothesis**.
        * There is a **0% chance** that the observed correlation is due to randomness.
        * The strong positive correlation is **statistically significant**.
<br><br>
* **Total Customer Spending vs. Average Spending per Order**:
    - Spearman's Rank Correlation Coefficient: **0.395**.
        * Indicates a **moderate positive monotonic relationship** between total customer spending and average spending per order.
    - P-value: **1.68e-59**
        * Since the p-value < 0.05, we **reject the null hypothesis**.
        * There is an **extremely low chance** (effectively 0%) that the observed correlation is due to randomness.
        * The moderate positive correlation is **statistically significant**.

#### Summary

Due to the non-normal distribution of the variables, we used **Spearman's Rank Correlation** to assess the direction, strength, and statistical significance of the **monotonic relationships** between total spending and the order frequency, as well as between total spending and average spending per order. Both correlations are **positive** and **statistically significant** at the 0.05 significance level, reinforcing the validity of observed relationships. We are now **confident** that:

* There is a **strong positive monotonic relationship** between customers' total spending and their order frequency. As customers place orders more frequently, their total spending tend to increase accordingly.

* There is a **moderate positive monotonic relationship** between customers' total spending and their average spending per order. Higher average spending per order is somewhat associated with increased total spending, but this relationship is not as strong as with the order frequency.

This analysis suggests that **repeat customers contribute more significantly to total sales** than customers who make high-value purchases per order. Therefore, implementing business strategies (e.g. loyalty program, repeat purchase incentives) aimed at **enhancing customer retention**, especially among the most loyal customers, and **encouraging repeat purchases**, is likely to be more effecitve in boosting total online sales than efforts focused on increasing their average spending per order.

<hr>

## Product Analysis

In [ ]:
# Load the cleaned pivoted full data parquet file into 
# a pandas DataFrame
df_pivoted = pd.read_parquet(
    './cafe_processed_files/cafe_full_pivoted_mx.parquet',
)

In [ ]:
df_products = (
    df_pivoted[
        [
            'order_id', 'customer_id', 'order_item_count', 
            'order_price', 'order_time', 'item_tracking_id', 'category', 
            'item', 'variation', 'quantity', 'item_price', 'unit_price', 
            'option_price'
        ]
    ].drop_duplicates()
    .sort_values(
        ['order_time', 'order_id', 'item_tracking_id'],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)

In [ ]:
df_products['order_year'] = df_products['order_time'].dt.year
df_products['order_month'] = df_products['order_time'].dt.month
df_products['order_day'] = df_products['order_time'].dt.day

In [ ]:
df_products.head()

### Top 5 Best-selling Items

In [ ]:
(
    df_products[
        ['item', 'item_price']
    ].groupby('item').sum()
    .rename(columns={'item_price': 'total_sales'})
    .sort_values('total_sales', ascending=False)
    .reset_index(drop=False)
).head()

In [ ]:
df_ = (
    df_products[
        ['category', 'item', 'item_price']
    ].groupby(['category', 'item']).sum()
    .rename(columns={'item_price': 'total_sales'})
    .reset_index(drop=False)
)

df_['rank'] = df_.sort_values(
        ['category', 'total_sales'], 
        ascending=[True, False],
    ).groupby(['category']).cumcount() + 1



df_[
    df_['rank'] <= 5
].sort_values(
    ['category', 'total_sales'], 
    ascending=[True, False],
).reset_index(drop=True)

<hr>

## Customer Analysis

## Products and Customer Churn Risk

### Identifying whether each unique order has a succeeding repeat purchase

For each order, the interval days until the next purchase order does NOT exceed the churn threshold, the next order is identified as a **repeat purchase**; otherwise, it is classified as a **new purchase**.

In [ ]:
# Select relevant columns for order analysis and sort the DataFrame
df_orders = df[
    ['customer_id', 'order_id', 'order_time', 'churn_threshold']
].sort_values(
    ['customer_id', 'order_time', 'order_id'],
    ascending=[True, True, True],
).drop_duplicates().reset_index(drop=True)

# Create a new column 'next_order_time' that holds the next 
# order time for each customer
df_orders['next_order_time'] = (
    df_orders.groupby(['customer_id'])['order_time'].shift(-1)
)

# Calculate the interval in days between the current order and the 
# previous order
df_orders['interval_to_next'] = df_orders.apply(
    lambda x: (x['next_order_time'] - x['order_time']).days,
    axis=1,
)

# Determine if the current purchase has a succeeding repeat purchase 
# based on the churn threshold
df_orders['next_repeat_purchase'] = df_orders.apply(
    lambda x: True if x['interval_to_next'] < x['churn_threshold']
    else False,
    axis=1,
)

In [ ]:
df_orders[
    df_orders['customer_id'] == 9
][
    ['order_id', 'interval_to_next', 'churn_threshold', 'next_repeat_purchase']
].head(10)

In [ ]:
df_orders[
    df_orders['customer_id'] == 9
]['next_repeat_purchase'].value_counts()

In [ ]:
# Select relevant columns for item analysis and sort the DataFrame
df_items = df[
    ['customer_id', 'order_id', 'order_time', 'item_tracking_id', 
     'item', 'quantity']
].sort_values(
    ['customer_id', 'order_time', 'order_id', 'item_tracking_id'],
    ascending=[True, True, True, True],
).drop_duplicates().reset_index(drop=True)

# Append the `next_repeat_purchase` column to the items data
df_items = df_items.merge(
    df_orders[['order_id', 'next_repeat_purchase']],
    how='inner',
    on='order_id',
)

In [ ]:
df_items[
    df_items['order_id'].isin([3449, 3509, 3521, 3533])
]

### Applying Bayes' Theorem to calculate the customer retention probability resulting from buying each particular item

To investigate whether buying a specific product would result in customer retention (or a repeat purchase), we can use **Bayes' Theorem** to determine the probability that a customer's next order is a repeat purchase given that they have purchased a specific item in their current order.

The **Bayes' Theorem** allows us to update our probability estimates for an event (next order being a repeat purchase) based on new evidence (the purchase of a specific item in the current order). The theorem in our case is methematically expressed as:

$$ P(repeat | product) = \frac{P(product | repeat) \times P(repeat)}{P(product)} $$

Where:
* **$P$(repeat | product)**: **Posterior Probability** - the probability that the next order is a repeat purchase given that the specific item was purchased in the current order.
* **$P$(product | repeat)**: **Likelihood** - the probability of purchasing the specific item given that the next order is a repeat purchase.
* **$P$(repeat)**: **Prior Probability** - the initial probability of the next order being a repeat purchase without considering the current purchase.
* **$P$(product)**: **Marginal Probability** - the probability of purchasing the specific item in the current order.

The reasons we apply **Bayes' Theorem** include:
1) Conditional Probability Estimation: The theorem is designed to compute conditional probabilities, which is exactly what's needed in this scenario: determining **$P$(repeat | product)**
2) Incorporating New Evidence: The purchase of a specific item serves as new evidence that can influence the likehood of a repeat purchase. Bayes' Theorem provides a structured way to incorporate this evidence into the probability assessment.
3) Flexibility with Prior Information: It allows the integration of prior knowledge or historical data ($P$(repeat)) with the new evidence ($P$(product | repeat)) to update the probability assessment.
4) Assumption of Independence: Bayes' Theorem assumes that the evidence (the purchase of a specific item in the current order) is conditionally independent of other factors given the hypothesis of next order being a repeat purchase.

In [ ]:
# Count the number of orders for each item that are followed by a 
# repeat purchase
df_repeat_item_count = (
    df_items[
        df_items['next_repeat_purchase']
    ][
        ['item', 'order_id']
    ].drop_duplicates().groupby('item').count()
    .reset_index(drop=False)
    .rename(columns={'order_id': 'item_order_count_next_repeat'})
)

# Count the total number of orders for each item
df_total_item_count = (
    df_items[
        ['item', 'order_id']
    ].drop_duplicates().groupby('item').count()
    .reset_index(drop=False)
    .rename(columns={'order_id': 'item_order_count'})
)

# Merge the total item purchase counts with the repeat purchase
# counts
df_item_count = df_total_item_count.merge(
    df_repeat_item_count,
    how='inner',
    on='item',
)

df_item_count.head()

To interpret the data in the DataFrame, Let's take the **(Entree) Pasta** item as an example. This item has been purchased in **13 orders** (`item_order_count`), **9 of which** have subsequent repeat purchases (`item_order_count_next_repeat`).

#### Calculate the estimated probability $P(repeat | product)$
$$ P(repeat | product) = \frac{P(product | repeat) \times P(repeat)}{P(product)} $$

We currently have the following data available:
* Total Number of Orders ($N_{total}$)
* Total Number of Orders for Each Item ($N_{item\_total}$)
* Number of Orders with a Subsequent Repeat Purchase for Each Item ($N_{item\_repeat}$)
* Total Number of Orders with a Subsequent Repeat Purchase ($N_{repeat}$)

The **Prior Probability** ($P(repeat)$), **Likelihood** ($P(product | repeat)$), and **Marginal Probability** ($P(product)$) for the Bayes' Theorem exression can be calculated as below shows:

$$P(repeat) = \frac {N_{repeat}} {N_{total}}$$

$$P(product) = \frac {N_{item\_total}} {N_{total}}$$

$$P(product | repeat) = \frac {N_{item\_repeat}} {N_{repeat}}$$

Plugging them into the **Bayes' Theorem exression**, we can get:

$$ P(repeat | product) = \frac{P(product | repeat) \times P(repeat)}{P(product)} 
    = \frac {(\frac {N_{item\_repeat}} {N_{repeat}}) \times (\frac {N_{repeat}} {N_{total}})} {(\frac {N_{item\_total}} {N_{total}})}
    = \frac {N_{item\_repeat}}{N_{item\_total}}
$$

Therefore, **the estimated probability that the next order is a repeat purchase given that a particular item was purchased in the current order** is equal to the **ratio** of **the number of orders with a subsequent repeat purchase for that Item** to **the total number of purchases of that item**.

### Smoothing Probability Estimates

The calculated probability $P$(repeat | product) could end up being exact 0 or 1 if none of the orders for a particular item has ever been followed by a repeat purchase, or if every order of that item has a subsequent repeat purchase. This can be misleading since the lack of historical data doesn't necessarily mean the repeat purchase event is impossible or guaranteed. 

To avoid $P$(repeat | product) being exactly 0 or 1, we can introduce **Smoothing** in the probability calculations. This approach can help prevent overconfidence in such cases, especially with limited sample sizes, providing more robust and realistic probability estimates.

Having introduced **Smoothing**, the Bayes' Theorem formula can now be expressed as:

$$ P(repeat | product) = \frac {N_{item\_repeat} + \alpha}{N_{item\_total} + 2\alpha} $$

Where:
* $\alpha$: Smoothing factor. Since we are dealing with binary outcomes (the next order is a repeat purchase or not), we need to add twice the smoothing factor to the denominator. This ensures that the total probability across both outcomes sums to 1, maintaining the balance between the two outcomes in the probability calculations.

While a smoothing factor of 1 is commonly used, it may be too large in our case, as the sample size (the number of orders for a particular item)  can be very small. Using a smaller smoothing factor, such as **0.01**, would introduce a smaller adjustment compared to 1, which better accounts for small sample sizes, while still preventing probabilities from being exactly 0 or 1.

In [ ]:
# Set the smoothing factor at 0.01
smoothing = 0.01

# Calculate the estimated customer retention probability given that
# a particular item was purchased in the current order
df_item_count['P_repeat_product'] = (
    (df_item_count['item_order_count_next_repeat'] + smoothing)
    / (df_item_count['item_order_count'] + 2 * smoothing)
)

df_item_count.head()

In [ ]:
print(df_item_count['P_repeat_product'].min())
print(df_item_count['P_repeat_product'].max())

The estimated probabilities now range from 0.251 to 0.999.

### Classifying Churn Risk of Products based on their Confidence Interval for Binomial Proportion

Given that the sample size (number of orders for each particular item) can be very **limited**, their individual probability estimates may be **unreliable** for the following reasons:
1) When the sample size is small, the **randomness** in the data becomes more pronounced. This makes it more likely that the estimated customer retention probabililty for an item is either overestimated or underestimated due to **chance occurrences** rather than a true reflection of customer behaviour for that item.
2) Small sample sizes make estimates **more susceptible** to **outliers** or **extreme values**. For example, if a product has been purchased in only a few orders, but none of them were followed by a repeat purchase, it may incorrectly suggest that buying the item would almost never lead to customer retention. This overconfident conclusion is based on insufficient data and is unlikely to generalize to the customers' true behaviour for that item. 
3) With a small sample size, data can become **highly sensitive to small changes** - even one or two additional order observations can markedly change the estimated probability. For example, if only 9 orders with no subsequent repeat purchases have been placed for a particular item, and a single repeat purchase occurs, the customer retention probability resulting from buying the item might jump from 0% to 10%. In contrast, with a larger sample size (say 1000 orders), adding one repeat purchase has much less impact, resulting in a more stable estimate.
4) The **Law of Large Numbers** states that as the sample size increases, the sample mean (or probability mean) will converge toward the true population mean. When the sample size is small, the estimated probability is more likely to **deviate from the true probability**, leading to a biased estimate of an item's true customer retention probability.

#### Calculating Wilson Score Interval

**Confidence Interval for Binomial Proportion** provides a range in which the true probability is likely to lie. With **smaller sample sizes**, the **confidence interval** becomes **wider**, reflecting **greater uncertainty** in the estimate. We can use the confidence interval to **quantify the reliability** of our estimates.

In our case, we will calculate **Wilson Score Interval** to assess the uncertainty in our estimated probabilities, as it doesn't depend on the normal approximation, making it more appropriate for small sample sizes. The items will then be categorised into different **churn risk levels** based on the **lower bound** and **upper bound** of the corresponding interval.

The lower and upper confidence bounds from the **Wilson Score Interval** are calculated as:

$$
\text{Lower Bound} = \frac{ \hat{p} + \frac{Z^2}{2n} - Z \sqrt{ \frac{ \hat{p}(1 - \hat{p}) }{n} + \frac{Z^2}{4n^2} } }{1 + \frac{Z^2}{n}}
$$

$$
\text{Upper Bound} = \frac{ \hat{p} + \frac{Z^2}{2n} + Z \sqrt{ \frac{ \hat{p}(1 - \hat{p}) }{n} + \frac{Z^2}{4n^2} } }{1 + \frac{Z^2}{n}}
$$


Where:
* $\hat{p} = \frac {x} {n}$ : The estimated probability that the next order is a repeat purchase given that a particular item was purchased in the current order.
* $x$: The number of orders with a subsequent repeat purchase for the item.
* $n$: The total number of orders for the item.
* $Z$: The Z-score for the desired confidence level (e.g. 1.96 for 95% confidence level).

In [ ]:
# Caculate the lower bound and upper bound of the Wilson
# Score Interval for each item
ci_lower, ci_upper = proportion_confint(
    count=df_item_count['item_order_count_next_repeat'],  # The number of orders with a subsequent repeat purchase for each item
    nobs=df_item_count['item_order_count'],  # The total number of orders for each item
    alpha=0.05,   # Significance level = 1 - confidence level. An alpha of 0.05 means the desired confidence level is 95%
    method='wilson',  # Wilson Score Interval method to use for confidence interval
)

df_item_count['ci_lower'] = ci_lower
df_item_count['ci_upper'] = ci_upper

In [ ]:
df_item_count.head()

#### Classifying Items based on Confidence Interval

Having calculated the Wilson Score Interval, we can categorise items into different churn risk levels based on the **lower bound** and **upper bound** of the interval.

In our analysis, we will use the **median** of the estimated probabilities across all items as a **threshold**, comparing it to the lower and upper bounds of each item's confidence interval to classify the **churn risk** of each item as **high**, **moderate**, and **low**, according to the following criteria:

* **Low Churn Risk Items**: Items whose **lower bound** of the confidence interval is **above** the threshold are classified as **High Retention Items**, or **Low Churn Risk Items**. Even at their lowest possible customer retention probability, the orders for these items have a relatively high chance of being followed by a repeat purchase, meaning that they present a **low risk** of contributing to customer churn.
* **High Churn Risk Items**: Items whose **upper bound** of the confidence interval falls **below** the threshold are classified as **Low Retention Items**, or **High Churn Risk Items**. Even at their highest possible customer retention probability, the orders for these items have a relatively low chance of being followed by a repeat purchase. This suggests that these items pose a **high risk** of contributing to customer churn.
* **Moderate Churn Risk Items**: Items with a lower bound below the threshold but an upper bound above the threshold are classified as **Moderate Retention Items**, or **Moderate Churn Risk Items**. This indicates that they carry a **moderate risk** of contributing to customer churn.

In [ ]:
# Calculate the median of the estimated probabilities across all items
median_prob = df_item_count['P_repeat_product'].median()

print(median_prob)

In [ ]:
# Classify each item's churn risk by comparing the lower 
# and upper bounds of their confidence interval against
# the median of estimates
def classify_churn_risk(x):
    if x['ci_lower'] > median_prob:
        return 'Low'
    elif x['ci_upper'] < median_prob:
        return 'High'
    else:
        return 'Moderate'

df_item_count['churn_risk_level'] = df_item_count.apply(
    classify_churn_risk, axis=1,
)

In [ ]:
df_item_count.head()

In [ ]:
# View the number of items in each churn risk category
df_item_count['churn_risk_level'].value_counts()

**19** products are classified as **High Churn Risk Item** (or **Low Retention Item**), **17** as **Low Churn Risk Item** (or **High Retention Item**), and **49** as **Moderate Churn Risk Item** (or **Moderate Retention Item**).

In [ ]:
# Create a new dataframe for items with high churn risk
items_high_churn_risk = df_item_count[
    df_item_count['churn_risk_level'] == 'High'
].sort_values('P_repeat_product', ascending=True)

# Create a new dataframe for items with high retention probability
items_high_retention = df_item_count[
    df_item_count['churn_risk_level'] == 'Low'
].sort_values('P_repeat_product', ascending=False)

In [ ]:
# Plot customer rentention probability over high retention items
plt.figure(figsize=(8, 6))

# Horizontal bar chart for customer rentention probability 
# over high retention items
sns.barplot(
    y=items_high_retention['item'], 
    x=items_high_retention['P_repeat_product'], 
    color='#8b4513',
    order=items_high_retention['item'],
)

for i in range(len(items_high_retention)):
    # Add Gantt-like bars for confidence intervals
    plt.hlines(
        y=i, 
        xmin=items_high_retention['ci_lower'].iloc[i], 
        xmax=items_high_retention['ci_upper'].iloc[i], 
        color='black', 
        linewidth=2,
    )
    
    # Add a dot at lower bound of each confidence interval
    plt.scatter(
        y=i, 
        x=items_high_retention['ci_lower'].iloc[i], 
        color='black',
        s=10,
    )
    
    # Add a dot at upper bound of each confidence interval
    plt.scatter(
        y=i,
        x=items_high_retention['ci_upper'].iloc[i],
        color='black',
        s=10,
    )

# Add a constant vertical line of the median of estimated probabilities
# at x-axis
plt.axvline(
    x=median_prob, 
    color='gray', 
    linestyle='--', 
    label='Median Customer Retention Probability',
)
    
# Add title and labels
plt.title(
    '\nItems with High Customer Retention Probability\n', 
    fontsize=14,
)
plt.legend()
plt.xlabel('Customer Retention Probability', fontsize=12)
plt.ylabel('')
plt.show()

# Display the dataframe for items with high retention probability
items_high_retention

In [ ]:
# Plot customer rentention probability over high churn risk items
plt.figure(figsize=(8, 6))

# Horizontal bar chart for customer rentention probability 
# over high churn risk items
sns.barplot(
    y=items_high_churn_risk['item'], 
    x=items_high_churn_risk['P_repeat_product'], 
    color='#f4d03f',
    order=items_high_churn_risk['item'],
)

for i in range(len(items_high_churn_risk)):
    # Add Gantt-like bars for confidence intervals
    plt.hlines(
        y=i, 
        xmin=items_high_churn_risk['ci_lower'].iloc[i], 
        xmax=items_high_churn_risk['ci_upper'].iloc[i], 
        color='black', 
        linewidth=2,
    )
    
    # Add a dot at lower bound of each confidence interval
    plt.scatter(
        y=i, 
        x=items_high_churn_risk['ci_lower'].iloc[i], 
        color='black',
        s=10,
    )
    
    # Add a dot at upper bound of each confidence interval
    plt.scatter(
        y=i,
        x=items_high_churn_risk['ci_upper'].iloc[i],
        color='black',
        s=10,
    )

# Add a constant vertical line of the median of estimated probabilities
# at x-axis
plt.axvline(
    x=median_prob, 
    color='gray', 
    linestyle='--', 
    label='Median Customer Retention Probability',
)

plt.title(
    '\nItems with High Customer Churn Risk\n', 
    fontsize=14,
)
plt.legend()
plt.xlabel('Customer Retention Probability', fontsize=12)
plt.ylabel('')
plt.show()

# Display the dataframe for items with high churn risk
items_high_churn_risk